# Gemma3-4B BP — Checkpoint-125 → GGUF Export

Bu notebook Drive'daki `checkpoint-125` LoRA adapter'ını base modele merge edip `q4_k_m` GGUF olarak dışa aktarır.

**Akış:**
1. Bağımlılıkları sabit sürümlere düşür (Transformers 4.46.3 + Unsloth 2024.12)
2. Runtime restart
3. Drive bağla
4. Checkpoint-125'i yükle
5. `merged_16bit` olarak kaydet
6. llama.cpp ile HF → GGUF (f16)
7. Q4_K_M quantize
8. Doğrulama

**Önkoşul:** Runtime → Change runtime type → **GPU (T4)**

## 1) Bağımlılıkları kur

Transformers 5.x'te Gemma3 multimodal save kırık. Bu hücre 4.46.3 + uyumlu Unsloth'u zorla yükler.

In [ ]:
!pip install -q --force-reinstall --no-deps "transformers==4.46.3"
!pip install -q --force-reinstall --no-deps "unsloth==2024.12.12" "unsloth-zoo==2024.12.7"
!pip install -q "accelerate>=0.34.0,<1.0" "peft>=0.13.0,<0.14" "tokenizers>=0.20,<0.21" "huggingface_hub>=0.26,<0.27" "trl>=0.12,<0.13" "bitsandbytes>=0.44"
print("\n>>> Kurulum bitti. Sonraki hücreyi çalıştırarak runtime'ı restart et.")

## 2) Runtime restart

Aşağıdaki hücre kernel'i öldürür. Restart sonrası **bu hücreyi tekrar çalıştırma** — 3. adımdan devam et.

In [ ]:
import os
os.kill(os.getpid(), 9)

## 3) Sürüm doğrula

Restart sonrası buradan devam et.

In [ ]:
import transformers, peft, accelerate
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("accelerate  :", accelerate.__version__)

try:
    import importlib.metadata as md
    print("unsloth     :", md.version("unsloth"))
    print("unsloth-zoo :", md.version("unsloth_zoo"))
except Exception as e:
    print("unsloth sürümü okunamadı:", e)

assert transformers.__version__.startswith("4.46"), "Transformers 4.46.x bekleniyordu"
print("\n>>> Sürümler OK, Hücre 4'e geç.")

## 4) Drive'ı bağla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5) Yolları tanımla

In [ ]:
import os

BASE_MODEL = "unsloth/gemma-3-4b-it"
CKPT       = "/content/drive/MyDrive/bp-finetune/checkpoints/checkpoint-125"
MERGED_DIR = "/content/drive/MyDrive/bp-finetune/merged/gemma3-4b-bp-v1"
GGUF_DIR   = "/content/drive/MyDrive/bp-finetune/gguf"
GGUF_F16   = f"{GGUF_DIR}/gemma3-4b-bp-v1.f16.gguf"
GGUF_Q4    = f"{GGUF_DIR}/gemma3-4b-bp-v1.q4_k_m.gguf"

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)

assert os.path.isdir(CKPT), f"Checkpoint bulunamadı: {CKPT}"
print("Checkpoint OK :", CKPT)
print("Merged hedef  :", MERGED_DIR)
print("GGUF hedef    :", GGUF_DIR)

## 6) Checkpoint-125'i yükle

Önce direkt checkpoint dizininden yüklemeyi dene (full PEFT model olarak kaydedildiyse). Hata olursa **6b** fallback'i çalıştır.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CKPT,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print("Checkpoint doğrudan yüklendi.")

### 6b) Fallback — base + adapter ayrı yükleme

Yukarıdaki hücre hata verirse bunu çalıştır.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model.load_adapter(CKPT)
print("Base + adapter yüklendi.")

## 7) Merged 16-bit kaydet

LoRA ağırlıklarını base'e gömüp standart HF formatında diske yaz.

In [ ]:
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)
print("Merged kaydedildi:", MERGED_DIR)
!ls -lh "$MERGED_DIR"

## 8) llama.cpp kur ve derle

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt

In [ ]:
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF
!cmake --build /content/llama.cpp/build --config Release -j --target llama-quantize

## 9) HF → GGUF (f16)

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
  "$MERGED_DIR" \
  --outfile "$GGUF_F16" \
  --outtype f16

## 10) Q4_K_M quantize

In [ ]:
!/content/llama.cpp/build/bin/llama-quantize \
  "$GGUF_F16" \
  "$GGUF_Q4" \
  Q4_K_M

## 11) Doğrula

In [ ]:
import os
for p in [GGUF_F16, GGUF_Q4]:
    if os.path.exists(p):
        gb = round(os.path.getsize(p) / 1e9, 2)
        print(f"OK  {p}  ->  {gb} GB")
    else:
        print(f"MISSING  {p}")

## 12) (Opsiyonel) f16 dosyasını sil — Drive'dan yer kazan

Q4_K_M tamamlandıysa f16 ara dosyasına gerek yok.

In [ ]:
# import os
# if os.path.exists(GGUF_F16):
#     os.remove(GGUF_F16)
#     print("Silindi:", GGUF_F16)